# 05 - Analyse Revenus & Pauvreté

Analyse des disparités de revenus et de pauvreté sur le territoire à partir des données FiloSoFi (INSEE, 2021).

In [10]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import DATA_PROCESSED, OUTPUTS

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 150
plt.rcParams["font.size"] = 10

In [11]:
filo_path = DATA_PROCESSED / "filosofi.parquet"
filo_dep_path = DATA_PROCESSED / "filosofi_par_departement.parquet"

if filo_path.exists() and filo_dep_path.exists():
    filo = pd.read_parquet(filo_path)
    filo_dep = pd.read_parquet(filo_dep_path)
    print(f"FiloSoFi communes: {filo.shape}")
    print(f"  Colonnes: {filo.columns.tolist()}")
    print(f"FiloSoFi d\u00e9partements: {filo_dep.shape}")
    print(f"  Colonnes: {filo_dep.columns.tolist()}")
else:
    print("Donn\u00e9es FiloSoFi non disponibles (t\u00e9l\u00e9ch\u00e9vement \u00e9chou\u00e9)")
    filo = None
    filo_dep = None

Données FiloSoFi non disponibles (téléchévement échoué)


In [12]:
if filo is not None:
    pauvrete_col = [c for c in filo.columns if "pauvreté" in c.lower()]
    niveau_vie_col = [c for c in filo.columns if "niv" in c.lower() or "médiane" in c.lower() or "revenu" in c.lower()]
    print(f"Colonnes pauvreté: {pauvrete_col}")
    print(f"Colonnes niveau de vie/revenu: {niveau_vie_col}")


## 5.1 Taux de pauvreté par département

In [13]:
if filo is not None:
    taux_pauvrete_col = [c for c in filo_dep.columns if "pauvreté" in c.lower()][0]
    mediane_revenu_col = [c for c in filo_dep.columns if "médiane" in c.lower()][0]

    print(f"Colonne taux pauvreté (dép): {taux_pauvrete_col}")
    print(f"Colonne médiane revenu (dép): {mediane_revenu_col}")


In [14]:
if filo is not None:
    fig, ax = plt.subplots(figsize=(10, 8))
    top_pauvrete = filo_dep.sort_values(taux_pauvrete_col, ascending=True).tail(30)

    colors = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(top_pauvrete)))
    ax.barh(top_pauvrete["code_departement"], top_pauvrete[taux_pauvrete_col], color=colors, edgecolor="white")
    ax.axvline(x=filo_dep[taux_pauvrete_col].median(), color="blue", linestyle="--", alpha=0.7,
               label=f"Médiane ({filo_dep[taux_pauvrete_col].median():.1f}%)")
    ax.set_xlabel("Taux de pauvreté (%)")
    ax.set_title("Top 30 départements par taux de pauvreté")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(OUTPUTS / "10_taux_pauvrete_departement.png", bbox_inches="tight")
    plt.show()


## 5.2 Niveau de vie médian par département

In [15]:
if filo is not None:
    fig, ax = plt.subplots(figsize=(10, 8))
    top_rev = filo_dep.sort_values(mediane_revenu_col, ascending=True).tail(30)

    colors = plt.cm.YlGn(np.linspace(0.3, 0.9, len(top_rev)))
    ax.barh(top_rev["code_departement"], top_rev[mediane_revenu_col], color=colors, edgecolor="white")
    ax.axvline(x=filo_dep[mediane_revenu_col].median(), color="red", linestyle="--", alpha=0.7,
               label=f"Médiane ({filo_dep[mediane_revenu_col].median():,.0f} €)")
    ax.set_xlabel("Niveau de vie médian (€/an)")
    ax.set_title("Top 30 départements par niveau de vie médian")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(OUTPUTS / "11_niveau_vie_departement.png", bbox_inches="tight")
    plt.show()


## 5.3 Corrélation pauvreté / logement social

In [16]:
if filo is not None:
    sru_dep = pd.read_parquet(DATA_PROCESSED / "sru_par_departement.parquet")

    merged = filo_dep[["code_departement", taux_pauvrete_col]].merge(
        sru_dep[["code_departement", "taux_sru_moyen"]], on="code_departement", how="inner"
    )

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(merged["taux_sru_moyen"], merged[taux_pauvrete_col], alpha=0.7, s=40, c="#e74c3c", edgecolors="white")

    mask = merged[["taux_sru_moyen", taux_pauvrete_col]].dropna()
    z = np.polyfit(mask["taux_sru_moyen"], mask[taux_pauvrete_col], 1)
    p = np.poly1d(z)
    x_line = np.linspace(mask["taux_sru_moyen"].min(), mask["taux_sru_moyen"].max(), 100)
    ax.plot(x_line, p(x_line), "b--", alpha=0.7, linewidth=1.5)

    corr = merged[["taux_sru_moyen", taux_pauvrete_col]].corr().iloc[0, 1]
    ax.set_xlabel("Taux moyen de logements sociaux (%)")
    ax.set_ylabel("Taux de pauvreté (%)")
    ax.set_title(f"Corrélation logement social vs pauvreté (r = {corr:.2f})")
    plt.tight_layout()
    plt.savefig(OUTPUTS / "12_correlation_sru_pauvrete.png", bbox_inches="tight")
    plt.show()


## 5.4 Corrélation pauvreté / éducation prioritaire

In [17]:
if filo is not None:
    edu_dep = pd.read_parquet(DATA_PROCESSED / "education_par_departement.parquet")

    merged2 = filo_dep[["code_departement", taux_pauvrete_col]].merge(
        edu_dep[["code_departement", "pct_ecoles_rep"]], on="code_departement", how="inner"
    )

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(merged2["pct_ecoles_rep"], merged2[taux_pauvrete_col], alpha=0.7, s=40, c="#9b59b6", edgecolors="white")

    mask2 = merged2[["pct_ecoles_rep", taux_pauvrete_col]].dropna()
    z2 = np.polyfit(mask2["pct_ecoles_rep"], mask2[taux_pauvrete_col], 1)
    p2 = np.poly1d(z2)
    x_line2 = np.linspace(mask2["pct_ecoles_rep"].min(), mask2["pct_ecoles_rep"].max(), 100)
    ax.plot(x_line2, p2(x_line2), "b--", alpha=0.7, linewidth=1.5)

    corr2 = merged2[["pct_ecoles_rep", taux_pauvrete_col]].corr().iloc[0, 1]
    ax.set_xlabel("% d'écoles en REP/REP+")
    ax.set_ylabel("Taux de pauvreté (%)")
    ax.set_title(f"Corrélation éducation prioritaire vs pauvreté (r = {corr2:.2f})")
    plt.tight_layout()
    plt.savefig(OUTPUTS / "13_correlation_ep_pauvrete.png", bbox_inches="tight")
    plt.show()


## Synthèse

In [18]:
if filo is not None:
    print("=== SYNTHESE REVENUS & PAUVRETE ===")
    print(f"Taux de pauvreté moyen  : {filo_dep[taux_pauvrete_col].mean():.1f}%")
    print(f"Taux de pauvreté médian : {filo_dep[taux_pauvrete_col].median():.1f}%")
    print(f"Niveau de vie médian moyen: {filo_dep[mediane_revenu_col].mean():,.0f} €")
    print(f"Dép. le + pauvre         : {top_pauvrete.iloc[-1]['code_departement']} ({top_pauvrete.iloc[-1][taux_pauvrete_col]:.1f}%)")
    print(f"Dép. le + riche          : {filo_dep.sort_values(mediane_revenu_col).iloc[-1]['code_departement']} ({filo_dep.sort_values(mediane_revenu_col).iloc[-1][mediane_revenu_col]:,.0f} €)")
    print(f"Corr. SRU/pauvreté       : r = {corr:.2f}")
    print(f"Corr. EP/pauvreté        : r = {corr2:.2f}")
